# MAYA_DC RD Curves and Fixed-Patch Reconstructions

This notebook is tailored to the `cedric-leonard/MAYA_DC` WandB project.
It does two things:

1. fetches runs from WandB, caches the large run table to CSV, filters MAYA_DC runs, aggregates seed statistics, and plots reusable RD curves;
2. loads one or more trained checkpoints from `run_output_dir/checkpoints/last.ckpt`, runs fixed-patch inference in-notebook, and displays `RCMC`, `RCMC_recon`, `SLC`, and `SLC_recon`.

The plotting/filtering helpers intentionally keep a few future-facing knobs around so the notebook still works if we later introduce new activations, training modes, or architectures.


## Imports and project setup


In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path
from typing import Any

import importlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb
import zarr
from IPython.display import display
from lightning import LightningModule
from maya4 import RC_MAX, RC_MIN, minmax_inverse
from omegaconf import DictConfig, OmegaConf, open_dict


def find_project_root(start: Path, marker: str = '.project-root') -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find project root from {start} using marker '{marker}'.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.processing_utils import (
    clip_mean_std_numpy,
    correct_swst_range_offset,
    phys_to_logI_np,
)
from src.utils.sarpyx_azimuth_compression import full_azimuth_compress_batch

NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'
CACHE_DIR = NOTEBOOK_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'CACHE_DIR:    {CACHE_DIR}')
print(f'DEVICE:       {DEVICE}')


## WandB cache configuration


In [ ]:
WANDB_ENTITY = 'cedric-leonard'
WANDB_PROJECT = 'MAYA_DC'

RUNS_CACHE_CSV = CACHE_DIR / f'wandb_runs_{WANDB_ENTITY}_{WANDB_PROJECT}.csv'
RD_STATS_CSV = CACHE_DIR / f'rd_stats_{WANDB_ENTITY}_{WANDB_PROJECT}.csv'
RD_STATS_WITH_BASELINES_CSV = CACHE_DIR / f'rd_stats_with_baselines_{WANDB_ENTITY}_{WANDB_PROJECT}.csv'
DEFAULT_BASELINE_SUMMARY_CSVS = [
    CACHE_DIR / 'jpeg_baseline_rcmc_channels_slc_summary.csv',
]

FORCE_REFRESH_RUN_CACHE = False
WANDB_API_TIMEOUT = 60
VERBOSE_FETCH = True

RUNS_CACHE_CSV


In [ ]:
MODEL_SHORT_NAMES = {
    'ScaleHyperprior': 'SHyp',
    'FactorizedPrior': 'FP',
}

DATASET_SHORT_NAMES = {
    'Maya4DataModule': 'MAYA4',
    'Maya4DirDataModule': 'MAYA4',
}

ACTIVATION_SHORT_NAMES = {
    'gdn': 'GDN',
    'relu': 'ReLU',
}

LOSS_SHORT_NAMES = {
    'CompoundCompressionLoss': 'Compound',
    'SimpleMSELoss': 'MSE',
}

METRIC_ALIASES = {
    'rate': ['test/rate', 'test/rate_epoch', 'test/bpp', 'test/bpp_epoch'],
    'psnr_amp': [
        'test/psnr_amp',
        'test/psnr_amp_epoch',
        'test/psnr_noisy',
        'test/psnr_noisy_epoch',
        'test/psnr_merlin',
        'test/psnr_merlin_epoch',
        'test/psnr_adam_noc',
        'test/psnr_adam_noc_epoch',
    ],
    'ssim_amp': [
        'test/ssim_amp',
        'test/ssim_amp_epoch',
        'test/ssim_noisy',
        'test/ssim_noisy_epoch',
        'test/ssim_merlin',
        'test/ssim_merlin_epoch',
        'test/ssim_adam_noc',
        'test/ssim_adam_noc_epoch',
    ],
    'complex_corr': [
        'test/complex_corr_mean',
        'test/complex_corr_mean_epoch',
        'test/coherence',
        'test/coherence_mean',
    ],
    'phase_err': ['test/phase_err_mean', 'test/phase_err_mean_epoch'],
}

PRETTY_METRIC_NAMES = {
    'test/rate': 'Bit-rate [bpp]',
    'test/rate_epoch': 'Bit-rate [bpp]',
    'test/bpp': 'Bit-rate [bpp]',
    'test/bpp_epoch': 'Bit-rate [bpp]',
    'test/psnr_amp': 'PSNR amplitude [dB]',
    'test/psnr_amp_epoch': 'PSNR amplitude [dB]',
    'test/psnr_noisy': 'PSNR amplitude [dB]',
    'test/ssim_amp': 'SSIM amplitude',
    'test/ssim_amp_epoch': 'SSIM amplitude',
    'test/complex_corr_mean': 'Complex correlation',
    'test/complex_corr_mean_epoch': 'Complex correlation',
    'test/phase_err_mean': 'Phase error',
    'test/phase_err_mean_epoch': 'Phase error',
}


def _cfg_select(cfg: DictConfig, key: str, default: Any = None) -> Any:
    value = OmegaConf.select(cfg, key)
    return default if value is None else value


def _json_dumps(value: Any) -> str:
    return json.dumps(value, default=str, sort_keys=True)


def _json_loads(value: Any, default: Any) -> Any:
    if isinstance(value, (dict, list)):
        return value
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return default
    if isinstance(value, str) and not value.strip():
        return default
    return json.loads(value)


def shorten_model_name(model_target: str | None) -> str:
    if not model_target:
        return 'UnknownModel'
    last = str(model_target).split('.')[-1]
    return MODEL_SHORT_NAMES.get(last, last)


def shorten_dataset_name(data_target: str | None) -> str:
    if not data_target:
        return 'UnknownDataset'
    last = str(data_target).split('.')[-1]
    return DATASET_SHORT_NAMES.get(last, last)


def shorten_activation_name(activation: str | None) -> str:
    if not activation:
        return 'UnknownAct'
    return ACTIVATION_SHORT_NAMES.get(str(activation).lower(), str(activation))


def shorten_loss_name(loss_target: str | None) -> str:
    if not loss_target:
        return 'UnknownLoss'
    last = str(loss_target).split('.')[-1]
    return LOSS_SHORT_NAMES.get(last, last)


def parse_run_name_metadata(run_name: str | None) -> dict[str, Any]:
    if not isinstance(run_name, str):
        return {}

    pattern = re.compile(
        r'^(?P<model_short>[^-_]+)-(?P<activation>[^_]+)'
        r'_s(?P<seed>[^_]+)'
        r'_L(?P<lmbda>[^_]+)'
        r'_(?P<training_mode>[^_]+)'
        r'_buf(?P<azimuth_buffer>[^_]+)'
        r'_(?P<criterion_short>[^_]+)'
        r'(?:_lr(?P<lr>[^_]+))?'
        r'(?:_b(?P<batch_size>[^_]+))?'
        r'(?:_(?P<patches_seen>\d+)p)?$'
    )
    match = pattern.match(run_name.strip())
    if match is None:
        return {}

    info = match.groupdict()
    model_short = info.get('model_short')
    activation = info.get('activation')
    criterion_short = info.get('criterion_short')

    return {
        'model_short': model_short,
        'model_name': {'SHP': 'ScaleHyperprior', 'FP': 'FactorizedPrior'}.get(model_short, model_short),
        'activation': activation,
        'activation_short': shorten_activation_name(activation),
        'seed': pd.to_numeric(info.get('seed'), errors='coerce'),
        'lmbda': pd.to_numeric(info.get('lmbda'), errors='coerce'),
        'training_mode': info.get('training_mode'),
        'azimuth_buffer': pd.to_numeric(info.get('azimuth_buffer'), errors='coerce'),
        'criterion_short': criterion_short,
        'criterion_target': criterion_short,
        'lr': pd.to_numeric(info.get('lr'), errors='coerce'),
        'batch_size': pd.to_numeric(info.get('batch_size'), errors='coerce'),
        'patches_seen': pd.to_numeric(info.get('patches_seen'), errors='coerce'),
    }


def infer_dataset_short(tags: list[str] | None, project: str | None = None) -> str:
    lowered_tags = {str(tag).lower() for tag in (tags or [])}
    if 'maya4' in lowered_tags or str(project) == 'MAYA_DC':
        return 'MAYA4'
    return 'UnknownDataset'


def fill_missing_scalar(current: Any, fallback: Any, unknown_tokens: set[str] | None = None) -> Any:
    unknown_tokens = unknown_tokens or set()
    if current is None:
        return fallback
    if isinstance(current, float) and pd.isna(current):
        return fallback
    if isinstance(current, str) and (not current.strip() or current in unknown_tokens):
        return fallback
    return current


def repair_run_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    repaired = df.copy()
    for column in [
        'training_mode',
        'model_short',
        'model_name',
        'activation',
        'activation_short',
        'criterion_short',
        'criterion_target',
        'dataset_short',
        'group_label',
    ]:
        if column not in repaired.columns:
            repaired[column] = None
        repaired[column] = repaired[column].astype(object)
    parsed_rows = repaired['run_name'].apply(parse_run_name_metadata)

    for idx, parsed in parsed_rows.items():
        if not parsed:
            continue
        for column in ['seed', 'lmbda', 'training_mode', 'azimuth_buffer']:
            repaired.at[idx, column] = fill_missing_scalar(repaired.at[idx, column], parsed.get(column))
        for column in ['model_short', 'model_name', 'activation', 'activation_short', 'criterion_short', 'criterion_target']:
            repaired.at[idx, column] = fill_missing_scalar(
                repaired.at[idx, column],
                parsed.get(column),
                unknown_tokens={'UnknownModel', 'UnknownAct', 'UnknownLoss'},
            )

    repaired['dataset_short'] = [
        fill_missing_scalar(current, infer_dataset_short(tags, project), {'UnknownDataset'})
        for current, tags, project in zip(
            repaired.get('dataset_short', pd.Series(index=repaired.index, dtype=object)),
            repaired['tags'],
            repaired['project'],
        )
    ]

    repaired['seed'] = pd.to_numeric(repaired['seed'], errors='coerce')
    repaired['lmbda'] = pd.to_numeric(repaired['lmbda'], errors='coerce')
    repaired['azimuth_buffer'] = pd.to_numeric(repaired['azimuth_buffer'], errors='coerce')
    repaired['group_label'] = repaired.apply(build_group_label, axis=1)
    return repaired


def build_group_label(row_like: dict[str, Any] | pd.Series) -> str:
    model_short = row_like.get('model_short', 'UnknownModel')
    activation_short = row_like.get('activation_short', 'UnknownAct')
    dataset_short = row_like.get('dataset_short', 'UnknownDataset')
    training_mode = row_like.get('training_mode', 'unknown_mode')
    parts = [
        f"{model_short}-{activation_short}",
        str(dataset_short),
        str(training_mode),
    ]
    no_output_padding = row_like.get('no_output_padding')
    if pd.notna(no_output_padding):
        parts.append('no_out_pad' if bool(no_output_padding) else 'out_pad')
    return ' | '.join(parts)


def extract_run_record(run: Any) -> dict[str, Any]:
    cfg = OmegaConf.create(run.config)
    summary = dict(run.summary._json_dict)
    tags = sorted(
        {
            str(tag)
            for tag in list(getattr(run, 'tags', []) or []) + list(_cfg_select(cfg, 'tags', []) or [])
        }
    )

    model_target = _cfg_select(cfg, 'model.net._target_', '')
    data_target = _cfg_select(cfg, 'data._target_', '')
    loss_target = _cfg_select(cfg, 'model.criterion._target_', '')
    activation = _cfg_select(cfg, 'model.net.activation', None)
    no_output_padding = _cfg_select(cfg, 'model.net.no_output_padding', None)

    record = {
        'id': run.id,
        'run_name': run.name,
        'state': getattr(run, 'state', None),
        'url': getattr(run, 'url', None),
        'created_at': getattr(run, 'created_at', None),
        'entity': WANDB_ENTITY,
        'project': WANDB_PROJECT,
        'seed': _cfg_select(cfg, 'seed', None),
        'lmbda': _cfg_select(cfg, 'model.criterion.lmbda', _cfg_select(cfg, 'lambda', None)),
        'training_mode': _cfg_select(cfg, 'model.training_mode', None),
        'azimuth_buffer': _cfg_select(cfg, 'azimuth_buffer', _cfg_select(cfg, 'model.azimuth_buffer', None)),
        'model_target': model_target,
        'model_name': str(model_target).split('.')[-1] if model_target else None,
        'model_short': shorten_model_name(model_target),
        'activation': activation,
        'activation_short': shorten_activation_name(activation),
        'data_target': data_target,
        'data_name': str(data_target).split('.')[-1] if data_target else None,
        'dataset_short': shorten_dataset_name(data_target),
        'criterion_target': loss_target,
        'criterion_short': shorten_loss_name(loss_target),
        'run_output_dir': _cfg_select(cfg, 'paths.output_dir', None),
        'no_output_padding': no_output_padding,
        'tags': tags,
        'group_label': None,  # filled below once all fields are present
        'summary': summary,
        'config': OmegaConf.to_container(cfg, resolve=False),
    }
    record['group_label'] = build_group_label(record)
    return record


def save_runs_cache(df: pd.DataFrame, cache_csv: Path) -> None:
    serializable = df.copy()
    for column in ['summary', 'config', 'tags']:
        serializable[column] = serializable[column].apply(_json_dumps)
    serializable.to_csv(cache_csv, index=False)
    print(f'Saved {len(serializable)} runs to {cache_csv}')


def load_runs_cache(cache_csv: Path) -> pd.DataFrame:
    cached = pd.read_csv(cache_csv)
    for column, default in [('summary', {}), ('config', {}), ('tags', [])]:
        cached[column] = cached[column].apply(lambda value: _json_loads(value, default))
    return repair_run_dataframe(cached)


def fetch_runs_dataframe(
    entity: str,
    project: str,
    cache_csv: Path,
    force_refresh: bool = False,
    timeout: int = 60,
    verbose: bool = True,
) -> pd.DataFrame:
    if cache_csv.exists() and not force_refresh:
        print(f'Loading cached runs from {cache_csv}')
        return load_runs_cache(cache_csv)

    api = wandb.Api(timeout=timeout)
    runs = api.runs(f'{entity}/{project}')
    records = [extract_run_record(run) for run in runs]
    df = pd.DataFrame.from_records(records)
    df = repair_run_dataframe(df)
    df.sort_values(['model_short', 'activation_short', 'training_mode', 'lmbda', 'seed'], inplace=True)
    save_runs_cache(df, cache_csv)
    if verbose:
        print(f'Fetched {len(df)} runs from {entity}/{project}')
    return df.reset_index(drop=True)


def collect_summary_keys(df: pd.DataFrame) -> list[str]:
    keys: set[str] = set()
    for summary in df['summary']:
        keys.update(summary.keys())
    return sorted(keys)


def metric_candidates(metric_name: str) -> list[str]:
    return METRIC_ALIASES.get(metric_name, [metric_name])


In [ ]:
runs_df = fetch_runs_dataframe(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    cache_csv=RUNS_CACHE_CSV,
    force_refresh=FORCE_REFRESH_RUN_CACHE,
    timeout=WANDB_API_TIMEOUT,
    verbose=VERBOSE_FETCH,
)

print(f'Loaded {len(runs_df)} runs.')
display(
    runs_df[
        [
            'id',
            'run_name',
            'model_short',
            'activation_short',
            'dataset_short',
            'training_mode',
            'criterion_short',
            'lmbda',
            'seed',
            'state',
        ]
    ].head()
)


## Filtering and experiment overview


In [ ]:
FILTER_CFG = {
    'seed_min': 0,
    'seed_max': 10,
    'blocked_tags': ['debug', 'crashed', 'BART'],
    # 'accepted_training_modes': ['slc'],
    # 'accepted_dataset_shorts': ['MAYA4'],
    # 'accepted_activation_shorts': ['GDN'],
    # 'accepted_model_shorts': ['SHyp'],
    # 'accepted_lambdas': [5, 7, 10, 20, 50, 100, 1000],
    # 'include_run_name_substrings': ['buf512'],
    # 'exclude_run_name_substrings': ['prototype'],
}


def print_removed_runs(title: str, removed_df: pd.DataFrame) -> None:
    if removed_df.empty:
        print(f'{title}: 0')
        return
    print(f'{title}: {len(removed_df)}')
    display(
        removed_df[
            [
                'id',
                'run_name',
                'seed',
                'lmbda',
                'model_short',
                'activation_short',
                'dataset_short',
                'training_mode',
                'tags',
            ]
        ]
    )


def apply_run_filters(df: pd.DataFrame, cfg: dict[str, Any], verbose: bool = True) -> pd.DataFrame:
    work = df.copy()
    work['seed'] = pd.to_numeric(work['seed'], errors='coerce')
    work['lmbda'] = pd.to_numeric(work['lmbda'], errors='coerce')

    seed_mask = work['seed'].between(cfg['seed_min'], cfg['seed_max'], inclusive='both')
    removed_seed = work.loc[~seed_mask].copy()
    work = work.loc[seed_mask].copy()
    if verbose:
        print_removed_runs(
            f"Removed runs outside seed range [{cfg['seed_min']}, {cfg['seed_max']}]",
            removed_seed,
        )

    blocked = [tag.lower() for tag in cfg.get('blocked_tags', [])]
    blocked_mask = work['tags'].apply(
        lambda tags: any(blocked_tag in str(tag).lower() for tag in (tags or []) for blocked_tag in blocked)
    )
    removed_tags = work.loc[blocked_mask].copy()
    work = work.loc[~blocked_mask].copy()
    if verbose:
        print_removed_runs('Removed runs with blocked tags', removed_tags)

    accepted_training_modes = cfg.get('accepted_training_modes')
    if accepted_training_modes is not None:
        work = work.loc[work['training_mode'].isin(accepted_training_modes)].copy()

    accepted_dataset_shorts = cfg.get('accepted_dataset_shorts')
    if accepted_dataset_shorts is not None:
        work = work.loc[work['dataset_short'].isin(accepted_dataset_shorts)].copy()

    accepted_activation_shorts = cfg.get('accepted_activation_shorts')
    if accepted_activation_shorts is not None:
        work = work.loc[work['activation_short'].isin(accepted_activation_shorts)].copy()

    accepted_model_shorts = cfg.get('accepted_model_shorts')
    if accepted_model_shorts is not None:
        work = work.loc[work['model_short'].isin(accepted_model_shorts)].copy()

    accepted_lambdas = cfg.get('accepted_lambdas')
    if accepted_lambdas is not None:
        work = work.loc[work['lmbda'].isin(accepted_lambdas)].copy()

    include_substrings = cfg.get('include_run_name_substrings')
    if include_substrings is not None:
        include_mask = work['run_name'].apply(
            lambda name: any(token.lower() in str(name).lower() for token in include_substrings)
        )
        work = work.loc[include_mask].copy()

    exclude_substrings = cfg.get('exclude_run_name_substrings')
    if exclude_substrings is not None:
        exclude_mask = work['run_name'].apply(
            lambda name: any(token.lower() in str(name).lower() for token in exclude_substrings)
        )
        work = work.loc[~exclude_mask].copy()

    work.sort_values(
        ['dataset_short', 'model_short', 'activation_short', 'training_mode', 'lmbda', 'seed'],
        inplace=True,
    )
    return work.reset_index(drop=True)


def default_group_columns(df: pd.DataFrame) -> list[str]:
    columns = ['model_short', 'activation_short', 'dataset_short', 'training_mode']
    if 'no_output_padding' in df.columns and df['no_output_padding'].nunique(dropna=True) > 1:
        columns.append('no_output_padding')
    return columns


def summarize_groups(df: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    summary = (
        df.groupby(group_columns, dropna=False)
        .agg(
            num_runs=('id', 'size'),
            num_lambdas=('lmbda', 'nunique'),
            num_seeds=('seed', 'nunique'),
            min_lambda=('lmbda', 'min'),
            max_lambda=('lmbda', 'max'),
            min_seed=('seed', 'min'),
            max_seed=('seed', 'max'),
        )
        .reset_index()
        .sort_values(group_columns)
    )
    return summary


filtered_runs_df = apply_run_filters(runs_df, FILTER_CFG, verbose=True)
GROUP_COLUMNS = default_group_columns(filtered_runs_df)

print(f'Kept {len(filtered_runs_df)} runs after filtering.')
display(
    summarize_groups(filtered_runs_df, GROUP_COLUMNS)
)

available_test_metrics = [key for key in collect_summary_keys(filtered_runs_df) if key.startswith('test/')]
print('Available test metrics:')
print(available_test_metrics)


## RD aggregation and plotting


In [ ]:
def resolve_metric_key(df: pd.DataFrame, metric_name: str) -> str:
    available_keys = set(collect_summary_keys(df))
    for candidate in metric_candidates(metric_name):
        if candidate in available_keys:
            return candidate
    raise KeyError(
        f"Could not resolve metric '{metric_name}'. Candidates: {metric_candidates(metric_name)}. "
        f"Available test metrics: {[key for key in sorted(available_keys) if key.startswith('test/')] }"
    )


def extract_metric_values(df: pd.DataFrame, metric_key: str) -> pd.Series:
    if metric_key in df.columns:
        return pd.to_numeric(df[metric_key], errors='coerce')
    return pd.to_numeric(df['summary'].apply(lambda summary: summary.get(metric_key)), errors='coerce')


def aggregate_rd_statistics(
    df: pd.DataFrame,
    rate_metric: str = 'rate',
    quality_metric: str = 'psnr_amp',
    group_columns: list[str] | None = None,
) -> tuple[pd.DataFrame, dict[str, str]]:
    group_columns = group_columns or default_group_columns(df)
    resolved_rate_metric = resolve_metric_key(df, rate_metric)
    resolved_quality_metric = resolve_metric_key(df, quality_metric)

    records: list[dict[str, Any]] = []
    for group_values, group_df in df.groupby(group_columns, dropna=False):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)
        group_meta = dict(zip(group_columns, group_values))
        group_meta['group_label'] = build_group_label(group_meta)

        for lmbda, lambda_df in group_df.groupby('lmbda', dropna=False):
            rate_values = extract_metric_values(lambda_df, resolved_rate_metric).dropna()
            quality_values = extract_metric_values(lambda_df, resolved_quality_metric).dropna()
            if rate_values.empty or quality_values.empty:
                continue

            record = {
                **group_meta,
                'lmbda': lmbda,
                'num_runs': len(lambda_df),
                'num_seed_values': lambda_df['seed'].nunique(dropna=True),
                'rate_mean': rate_values.mean(),
                'rate_std': rate_values.std(ddof=0),
                'rate_min': rate_values.min(),
                'rate_max': rate_values.max(),
                'quality_mean': quality_values.mean(),
                'quality_std': quality_values.std(ddof=0),
                'quality_min': quality_values.min(),
                'quality_max': quality_values.max(),
            }
            records.append(record)

    stats_df = pd.DataFrame.from_records(records)
    if stats_df.empty:
        raise ValueError('No statistics could be computed for the requested plot metrics after filtering.')

    stats_df.sort_values(group_columns + ['lmbda'], inplace=True)
    return stats_df.reset_index(drop=True), {
        'rate_metric': resolved_rate_metric,
        'quality_metric': resolved_quality_metric,
    }


def pretty_metric_name(metric_key: str) -> str:
    return PRETTY_METRIC_NAMES.get(metric_key, metric_key)


def format_lambda(value: float) -> str:
    if pd.isna(value):
        return 'nan'
    rounded = round(float(value))
    return str(int(rounded)) if abs(float(value) - rounded) < 1e-9 else f'{float(value):g}'


def build_single_group_title(stats_df: pd.DataFrame) -> str:
    row = stats_df.iloc[0]
    title_parts = [f"{row['model_short']}-{row['activation_short']}", row['dataset_short'], str(row['training_mode'])]
    if 'no_output_padding' in stats_df.columns and pd.notna(row.get('no_output_padding')):
        title_parts.append('no_out_pad' if bool(row['no_output_padding']) else 'out_pad')
    return ' | '.join(title_parts)


def plot_rd_curve(
    stats_df: pd.DataFrame,
    resolved_rate_metric: str,
    resolved_quality_metric: str,
    show_rate_error_bars: bool = True,
    show_quality_error_bars: bool = False,
    show_quality_band: bool = True,
    annotate_lambda: bool = True,
    title: str | None = None,
    show_legend: bool | None = None,
    ax: plt.Axes | None = None,
) -> tuple[plt.Figure, plt.Axes]:
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 6))
    else:
        fig = ax.figure

    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    activation_linestyles = {
        'GDN': '-',
        'ReLU': '--',
    }
    training_mode_markers = {
        'slc': 'o',
        'rcmc': 's',
    }

    unique_groups = stats_df['group_label'].nunique()
    if show_legend is None:
        show_legend = unique_groups > 1
    if title is None and unique_groups == 1:
        title = build_single_group_title(stats_df)
    if title is None:
        title = 'RD curve'

    for color_idx, (group_label, group_df) in enumerate(stats_df.groupby('group_label', sort=False)):
        group_df = group_df.sort_values('lmbda')
        row0 = group_df.iloc[0]
        linestyle = activation_linestyles.get(row0['activation_short'], '-')
        marker = training_mode_markers.get(str(row0['training_mode']), 'o')
        color = colors[color_idx % len(colors)]

        ax.plot(
            group_df['rate_mean'],
            group_df['quality_mean'],
            marker=marker,
            linestyle=linestyle,
            color=color,
            linewidth=1.8,
            markersize=6,
            label=group_label,
        )

        if show_quality_band:
            ax.fill_between(
                group_df['rate_mean'],
                group_df['quality_min'],
                group_df['quality_max'],
                color=color,
                alpha=0.15,
            )

        if show_rate_error_bars or show_quality_error_bars:
            ax.errorbar(
                group_df['rate_mean'],
                group_df['quality_mean'],
                xerr=group_df['rate_std'] if show_rate_error_bars else None,
                yerr=group_df['quality_std'] if show_quality_error_bars else None,
                fmt='none',
                ecolor=color,
                elinewidth=1.0,
                capsize=2,
                alpha=0.9,
            )

        if annotate_lambda:
            for _, point in group_df.iterrows():
                ax.annotate(
                    format_lambda(point['lmbda']),
                    (point['rate_mean'], point['quality_mean']),
                    textcoords='offset points',
                    xytext=(5, 5),
                    fontsize=9,
                    color=color,
                )

    ax.set_title(title)
    ax.set_xlabel(pretty_metric_name(resolved_rate_metric))
    ax.set_ylabel(pretty_metric_name(resolved_quality_metric))
    ax.grid(True, alpha=0.3)
    if show_legend:
        ax.legend(loc='best')
    plt.tight_layout()
    return fig, ax


In [ ]:
RD_PLOT_CFG = {
    'rate_metric': 'rate',
    'quality_metric': 'psnr_amp',
    'group_columns': GROUP_COLUMNS,
    'show_rate_error_bars': True,
    'show_quality_error_bars': False,
    'show_quality_band': True,
    'annotate_lambda': True,
    'title': None,
    'save_stats_csv': True,
    'baseline_summary_csvs': DEFAULT_BASELINE_SUMMARY_CSVS,
    'save_combined_stats_csv': True,
}


def load_baseline_summary_csvs(paths: list[Path]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for path in paths:
        path = Path(path)
        if not path.exists():
            print(f'Skipping missing baseline summary CSV: {path}')
            continue
        baseline_df = pd.read_csv(path)
        if baseline_df.empty:
            print(f'Skipping empty baseline summary CSV: {path}')
            continue
        baseline_df['curve_source'] = baseline_df.get('curve_source', 'classical_codec')
        baseline_df['baseline_summary_csv'] = str(path)
        frames.append(baseline_df)
        print(f'Loaded baseline summary CSV: {path}')

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, sort=False)


rd_stats_df, resolved_metrics = aggregate_rd_statistics(
    filtered_runs_df,
    rate_metric=RD_PLOT_CFG['rate_metric'],
    quality_metric=RD_PLOT_CFG['quality_metric'],
    group_columns=RD_PLOT_CFG['group_columns'],
)

if RD_PLOT_CFG['save_stats_csv']:
    rd_stats_df.to_csv(RD_STATS_CSV, index=False)
    print(f'Saved RD statistics to {RD_STATS_CSV}')

baseline_rd_stats_df = load_baseline_summary_csvs(RD_PLOT_CFG['baseline_summary_csvs'])
combined_rd_stats_df = rd_stats_df.copy()
if not baseline_rd_stats_df.empty:
    combined_rd_stats_df = pd.concat([combined_rd_stats_df, baseline_rd_stats_df], ignore_index=True, sort=False)

if RD_PLOT_CFG['save_combined_stats_csv']:
    combined_rd_stats_df.to_csv(RD_STATS_WITH_BASELINES_CSV, index=False)
    print(f'Saved combined RD statistics to {RD_STATS_WITH_BASELINES_CSV}')

display(rd_stats_df)
if not baseline_rd_stats_df.empty:
    display(baseline_rd_stats_df)

fig, ax = plot_rd_curve(
    combined_rd_stats_df,
    resolved_rate_metric=resolved_metrics['rate_metric'],
    resolved_quality_metric=resolved_metrics['quality_metric'],
    show_rate_error_bars=RD_PLOT_CFG['show_rate_error_bars'],
    show_quality_error_bars=RD_PLOT_CFG['show_quality_error_bars'],
    show_quality_band=RD_PLOT_CFG['show_quality_band'],
    annotate_lambda=RD_PLOT_CFG['annotate_lambda'],
    title=RD_PLOT_CFG['title'],
)
plt.show()


## Fixed-patch reconstruction visualization


In [ ]:
RECONSTRUCTION_CFG = {
    'run_identifiers': [
        'SHP-gdn_s0_L1000_slc_buf512_Compound_lr0.0001_b4_10000p',
    ],
    'patch_json': PROJECT_ROOT / 'data/fixed_patches/s1c-s6-raw-s-vv-20250422t061414-20250422t061446-002002-0040d4.json',
    'checkpoint_name': 'last.ckpt',
    'clip_factor': 3.0,
    'force_patch_version': False,
}

RECONSTRUCTION_CFG


In [ ]:
def resolve_existing_path(path_like: str | Path | None) -> Path | None:
    if path_like is None or (isinstance(path_like, float) and pd.isna(path_like)):
        return None
    path = Path(path_like)
    candidates = [path]
    if not path.is_absolute():
        candidates.append(PROJECT_ROOT / path)
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return None


def find_run_row(df: pd.DataFrame, identifier: str) -> pd.Series:
    exact = df[(df['id'] == identifier) | (df['run_name'] == identifier)]
    if len(exact) == 1:
        return exact.iloc[0]
    if len(exact) > 1:
        exact = exact.sort_values('created_at')
        print(f"Multiple runs matched '{identifier}'. Using the most recent one.")
        return exact.iloc[-1]

    partial = df[df['run_name'].str.contains(identifier, case=False, na=False)]
    if len(partial) == 1:
        print(f"Using partial run_name match for '{identifier}': {partial.iloc[0]['run_name']}")
        return partial.iloc[0]

    if partial.empty:
        raise KeyError(f"Could not find run '{identifier}'.")

    raise KeyError(
        f"Run identifier '{identifier}' is ambiguous. Matching run names: {partial['run_name'].tolist()}"
    )


def resolve_run_output_dir(run_row: pd.Series) -> Path:
    run_output_dir = resolve_existing_path(run_row.get('run_output_dir'))
    if run_output_dir is None:
        raise FileNotFoundError(
            f"Could not resolve run_output_dir for run '{run_row['run_name']}' (id={run_row['id']})."
        )
    return run_output_dir


def find_checkpoint_path(run_row: pd.Series, checkpoint_name: str = 'last.ckpt') -> Path:
    run_output_dir = resolve_run_output_dir(run_row)
    checkpoint_dir = run_output_dir / 'checkpoints'
    checkpoint_path = checkpoint_dir / checkpoint_name
    if checkpoint_path.exists():
        return checkpoint_path.resolve()

    fallback_ckpts = sorted(checkpoint_dir.glob('*.ckpt')) if checkpoint_dir.exists() else []
    if fallback_ckpts:
        print(
            f"Checkpoint '{checkpoint_name}' was not found for {run_row['run_name']}. "
            f"Using '{fallback_ckpts[-1].name}' instead."
        )
        return fallback_ckpts[-1].resolve()

    raise FileNotFoundError(
        f"No checkpoint was found under {checkpoint_dir} for run '{run_row['run_name']}'."
    )


def harmonize_cfg_for_loading(hydra_cfg: DictConfig) -> DictConfig:
    with open_dict(hydra_cfg):
        patch_size = OmegaConf.select(hydra_cfg, 'patch_size')
        azimuth_buffer = OmegaConf.select(hydra_cfg, 'azimuth_buffer')
        if patch_size is not None and OmegaConf.select(hydra_cfg, 'model.criterion.patch_size') is None:
            hydra_cfg.model.criterion.patch_size = patch_size
        if azimuth_buffer is not None:
            if OmegaConf.select(hydra_cfg, 'model.criterion.azimuth_buffer') is None:
                hydra_cfg.model.criterion.azimuth_buffer = azimuth_buffer
            if OmegaConf.select(hydra_cfg, 'model.azimuth_buffer') is None:
                hydra_cfg.model.azimuth_buffer = azimuth_buffer
    return hydra_cfg


def load_hydra_cfg_for_run(run_row: pd.Series) -> DictConfig:
    run_output_dir = resolve_run_output_dir(run_row)
    hydra_cfg_path = run_output_dir / '.hydra' / 'config.yaml'
    if hydra_cfg_path.exists():
        cfg = OmegaConf.load(hydra_cfg_path)
        try:
            OmegaConf.resolve(cfg)
        except Exception as exc:
            print(f"OmegaConf.resolve() raised {type(exc).__name__}: {exc}. Continuing with unresolved values.")
        return harmonize_cfg_for_loading(cfg)

    print(
        f"No .hydra/config.yaml found for {run_row['run_name']}. Falling back to the cached WandB config."
    )
    cfg = OmegaConf.create(run_row['config'])
    try:
        OmegaConf.resolve(cfg)
    except Exception as exc:
        print(f"OmegaConf.resolve() raised {type(exc).__name__}: {exc}. Continuing with unresolved values.")
    return harmonize_cfg_for_loading(cfg)


def locate_symbol(target: str) -> Any:
    module_name, symbol_name = target.rsplit('.', 1)
    module = importlib.import_module(module_name)
    return getattr(module, symbol_name)


def instantiate_from_target_config(config: Any) -> Any:
    if isinstance(config, DictConfig):
        try:
            config = OmegaConf.to_container(config, resolve=True)
        except Exception as exc:
            print(f"OmegaConf.to_container(resolve=True) raised {type(exc).__name__}: {exc}. Falling back to resolve=False.")
            config = OmegaConf.to_container(config, resolve=False)
    if isinstance(config, list):
        return [instantiate_from_target_config(item) for item in config]
    if not isinstance(config, dict):
        return config

    kwargs = {
        key: instantiate_from_target_config(value)
        for key, value in config.items()
        if not str(key).startswith('_')
    }
    target = config.get('_target_')
    if target is None:
        return kwargs

    cls_or_fn = locate_symbol(target)
    if config.get('_partial_', False):
        from functools import partial

        return partial(cls_or_fn, **kwargs)
    return cls_or_fn(**kwargs)


def instantiate_model_and_load_weights(
    hydra_cfg: DictConfig,
    ckpt_path: Path,
    force_patch_version: bool = False,
) -> LightningModule:
    print(f"Instantiating model <{hydra_cfg.model._target_}>")
    model: LightningModule = instantiate_from_target_config(hydra_cfg.model)
    checkpoint = torch.load(str(ckpt_path), map_location='cpu')
    msg = model.load_state_dict(checkpoint['state_dict'], strict=True)
    print(f'Loaded checkpoint state_dict with message: {msg}')
    model.to(DEVICE)
    model.eval()
    return model

def resolve_patch_json_path(patch_json: str | Path) -> Path:
    patch_json = Path(patch_json)
    candidates = [
        patch_json,
        PROJECT_ROOT / patch_json,
        PROJECT_ROOT / 'data' / 'fixed_patches' / patch_json.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Patch JSON not found for {patch_json}.')


def resolve_product_path(product: str, json_path: Path, hydra_cfg: DictConfig) -> Path:
    raw_path = Path(product)
    candidates = []
    if raw_path.is_absolute():
        candidates.append(raw_path)
    else:
        candidates.extend([json_path.parent / raw_path, PROJECT_ROOT / raw_path, raw_path])

    for key in ('train_dir', 'val_dir', 'test_dir'):
        value = OmegaConf.select(hydra_cfg, f'data.{key}')
        if value:
            root = Path(str(value)).expanduser()
            candidates.extend([root / raw_path.name, root.parent / raw_path.name])

    seen: set[Path] = set()
    for candidate in candidates:
        resolved = candidate.expanduser()
        if resolved in seen:
            continue
        seen.add(resolved)
        if resolved.exists():
            return resolved.resolve()

    raise FileNotFoundError(
        f"Could not resolve product '{product}' from patch JSON '{json_path}'."
    )


def load_complex_patch(
    arr: zarr.Array,
    az_sl: slice,
    rg_sl: slice,
    norm_constants: tuple[float, float] | None = None,
) -> np.ndarray:
    data = np.array(arr[az_sl, rg_sl], dtype=np.complex128)
    stacked = np.stack([np.real(data), np.imag(data)], axis=0).astype(np.float32)
    if norm_constants is not None:
        vmin, vmax = norm_constants
        stacked = (stacked - vmin) / (vmax - vmin)
    return stacked


def load_fixed_patch_bundle(patch_json: str | Path, hydra_cfg: DictConfig) -> dict[str, Any]:
    json_path = resolve_patch_json_path(patch_json)
    patch_info = json.loads(json_path.read_text())
    zarr_path = resolve_product_path(patch_info['product'], json_path, hydra_cfg)

    az_start = int(patch_info['az_start'])
    az_end = int(patch_info['az_end'])
    rg_start = int(patch_info['rg_start'])
    rg_end = int(patch_info['rg_end'])
    buffer_size = int(OmegaConf.select(hydra_cfg, 'azimuth_buffer'))
    az_full_start = az_start - buffer_size
    az_full_end = az_end + buffer_size
    if az_full_start < 0:
        raise ValueError(
            f"Patch starts too close to the top edge: az_start={az_start}, buffer={buffer_size}."
        )

    rcmc_arr = zarr.open_array(str(zarr_path / 'rcmc'), mode='r')
    az_arr = zarr.open_array(str(zarr_path / 'az'), mode='r')

    rcmc_norm = torch.from_numpy(
        load_complex_patch(
            rcmc_arr,
            slice(az_full_start, az_full_end),
            slice(rg_start, rg_end),
            norm_constants=(RC_MIN, RC_MAX),
        )
    ).unsqueeze(0)

    rcmc_phys = load_complex_patch(
        rcmc_arr,
        slice(az_start, az_end),
        slice(rg_start, rg_end),
    )
    slc_phys = load_complex_patch(
        az_arr,
        slice(az_start, az_end),
        slice(rg_start, rg_end),
    )

    store_root = zarr.open_group(str(zarr_path), mode='r')
    metadata_rows = []
    ephemeris_rows = []
    if 'metadata' in store_root.attrs:
        raw = store_root.attrs['metadata']
        if isinstance(raw, dict) and isinstance(raw.get('data'), list):
            metadata_rows = raw['data']
    if 'ephemeris' in store_root.attrs:
        raw = store_root.attrs['ephemeris']
        if isinstance(raw, dict) and isinstance(raw.get('data'), list):
            ephemeris_rows = raw['data']

    metadata_full = pd.DataFrame(metadata_rows)
    ephemeris_df = pd.DataFrame(ephemeris_rows)
    az_clip_end = min(az_full_end, len(metadata_full))
    metadata_patch = metadata_full.iloc[az_full_start:az_clip_end].reset_index(drop=True).copy()
    if not metadata_patch.empty:
        correct_swst_range_offset(metadata_patch, rg_start)

    return {
        'patch_stem': json_path.stem,
        'zarr_path': zarr_path,
        'buffer_size': buffer_size,
        'az_core': az_end - az_start,
        'rcmc_norm': rcmc_norm,
        'rcmc_phys': rcmc_phys,
        'slc_phys': slc_phys,
        'metadata': metadata_patch if not metadata_patch.empty else None,
        'ephemeris': ephemeris_df if not ephemeris_df.empty else None,
        'coords': {'zfile': str(zarr_path), 'y': az_full_start, 'x': rg_start},
    }


def run_fixed_patch_inference(
    model: LightningModule,
    patch_bundle: dict[str, Any],
    clip_factor: float = 3.0,
    try_slc_reconstruction: bool = True,
) -> dict[str, Any]:
    with torch.no_grad():
        output = model(patch_bundle['rcmc_norm'].to(DEVICE))
        x_hat = output.x_hat

    buffer_size = patch_bundle['buffer_size']
    az_core = patch_bundle['az_core']
    x_hat_core = x_hat[:, :, buffer_size : buffer_size + az_core, :]
    x_hat_core_phys = minmax_inverse(x_hat_core, RC_MIN, RC_MAX).squeeze(0).cpu().numpy()

    slc_recon_phys = None
    if (
        try_slc_reconstruction
        and patch_bundle['metadata'] is not None
        and patch_bundle['ephemeris'] is not None
    ):
        # Unlike the training callback, the notebook always tries SLC reconstruction when metadata exists.
        with torch.no_grad():
            x_hat_phys = minmax_inverse(x_hat, RC_MIN, RC_MAX)
            slc_recon = full_azimuth_compress_batch(
                x_hat_phys,
                [patch_bundle['metadata']],
                [patch_bundle['ephemeris']],
                buffer_size=buffer_size,
                device=str(DEVICE),
                coords_batch=[patch_bundle['coords']],
            )
        slc_recon_phys = slc_recon.squeeze(0).cpu().numpy()

    return {
        'patch_stem': patch_bundle['patch_stem'],
        'clip_factor': clip_factor,
        'rcmc': patch_bundle['rcmc_phys'],
        'rcmc_recon': x_hat_core_phys,
        'slc': patch_bundle['slc_phys'],
        'slc_recon': slc_recon_phys,
    }


def to_logi_image(array_2ch: np.ndarray, clip_factor: float) -> np.ndarray:
    return clip_mean_std_numpy(phys_to_logI_np(array_2ch), clip_factor).T


def plot_reconstruction_results(results: list[dict[str, Any]]) -> tuple[plt.Figure, np.ndarray]:
    column_keys = ['rcmc', 'rcmc_recon', 'slc', 'slc_recon']
    column_titles = ['RCMC', 'RCMC_recon', 'SLC', 'SLC_recon']
    fig, axes = plt.subplots(
        len(results),
        len(column_keys),
        figsize=(4.5 * len(column_keys), 4.2 * len(results)),
        squeeze=False,
    )

    for row_idx, result in enumerate(results):
        row_label = result['run_name']
        for col_idx, (column_key, column_title) in enumerate(zip(column_keys, column_titles)):
            ax = axes[row_idx, col_idx]
            ax.set_facecolor('black')
            array = result[column_key]
            if array is None:
                ax.text(0.5, 0.5, 'Unavailable', ha='center', va='center', color='white', fontsize=11)
                ax.set_xticks([])
                ax.set_yticks([])
            else:
                image = to_logi_image(array, result['clip_factor'])
                ax.imshow(image, cmap='viridis', aspect='auto', origin='upper')
                ax.set_xticks([])
                ax.set_yticks([])
            if row_idx == 0:
                ax.set_title(column_title)
            if col_idx == 0:
                ax.set_ylabel(row_label)

    patch_stem = results[0]['patch_stem'] if results else 'unknown_patch'
    fig.suptitle(f"Fixed patch: {patch_stem}", fontsize=12)
    plt.tight_layout()
    return fig, axes


def visualize_fixed_patch_reconstructions(
    run_identifiers: list[str],
    patch_json: str | Path,
    checkpoint_name: str = 'last.ckpt',
    clip_factor: float = 3.0,
    force_patch_version: bool = False,
) -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    for identifier in run_identifiers:
        run_row = find_run_row(runs_df, identifier)
        hydra_cfg = load_hydra_cfg_for_run(run_row)
        ckpt_path = find_checkpoint_path(run_row, checkpoint_name=checkpoint_name)
        patch_bundle = load_fixed_patch_bundle(patch_json, hydra_cfg)
        model = instantiate_model_and_load_weights(
            hydra_cfg,
            ckpt_path,
            force_patch_version=force_patch_version,
        )
        inference = run_fixed_patch_inference(model, patch_bundle, clip_factor=clip_factor)
        results.append(
            {
                **inference,
                'run_id': run_row['id'],
                'run_name': run_row['run_name'],
                'checkpoint_path': str(ckpt_path),
            }
        )
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return results


In [ ]:
reconstruction_results = visualize_fixed_patch_reconstructions(
    run_identifiers=RECONSTRUCTION_CFG['run_identifiers'],
    patch_json=RECONSTRUCTION_CFG['patch_json'],
    checkpoint_name=RECONSTRUCTION_CFG['checkpoint_name'],
    clip_factor=RECONSTRUCTION_CFG['clip_factor'],
    force_patch_version=RECONSTRUCTION_CFG['force_patch_version'],
)

display(
    pd.DataFrame(
        {
            'run_name': [result['run_name'] for result in reconstruction_results],
            'run_id': [result['run_id'] for result in reconstruction_results],
            'checkpoint_path': [result['checkpoint_path'] for result in reconstruction_results],
        }
    )
)

fig, axes = plot_reconstruction_results(reconstruction_results)
plt.show()
